In [22]:
import cv2
import pytesseract
import pandas as pd
import re
from datetime import datetime
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk
import numpy as np

In [23]:
class SalesReportExtractor:
    def __init__(self):
        self.day_mapping ={
            'monday': 1, 'tuesday' : 2, 'wednesday' : 3, 'thursday' : 4,
            'friday' : 5, 'saturday' : 6, 'sunday' : 7, 
            'mon': 1, 'tue' : 2, 'wed' : 3, 'thurs' : 4,
            'fri' : 5, 'sat' : 6, 'sun' : 7,
        }

        self.reason_mapping = {
            'f': 'Facebook', 'fb': 'Facebook',
            'm': 'Marketing',
            'live': 'Livestream', 'l' : 'Livestream',
            'r1' : 'Repeat Customer',
            'r2' : 'Referral',
            'w' : 'Walk-in',
        }

        self.extracted_data = []
        self.unclear_markings = set()

In [24]:
def preprocess_image(self, image_path):
    # Read image
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply denoising
    denoised = cv2.fastNlMeansDenoising()

    # Apply threshold to get binary image
    _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Morphological operations to clean up
    kernel = np.ones((1,1), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    return cleaned

In [25]:
def extract_text_from_image(self, image_path):
    """Extracting text from image using OCR"""
    try:
        processed_img = self.preprocess_image(image_path)
        custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz()/-:., '
        text = pytesseract.image_to_string(processed_img)
        return text
    except Exception as e:
        print(f"Error extracting text from {image_path}")
        return ""

In [26]:
def parse_date_and_day(self, text):
        """Extract date and day from text"""
        date_pattern = r'Date\s*:?\s*(\d{1,2})/(\d{1,2})/(\d{4})\s*\(([^)]+)\)'
        match = re.search(date_pattern, text, re.IGNORECASE)
        
        if match:
            day, month, year = match.groups()[:3]
            day_text = match.group(4).strip().lower()
            
            # Parse date
            try:
                date_obj = datetime.strptime(f"{day}/{month}/{year}", "%d/%m/%Y")
                formatted_date = date_obj.strftime("%Y-%m-%d")
            except:
                formatted_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}"
            
            # Get day number
            day_num = None
            for day_key, num in self.day_mapping.items():
                if day_key in day_text:
                    day_num = num
                    break
            
            return formatted_date, day_text.title(), day_num
        
        return None, None, None

In [27]:
def parse_sales_data(self, text):
        """Parse individual sales entries from text"""
        lines = text.split('\n')
        sales_data = []
        
        # Look for table data (rows with S/N, Name, Sales Order, Amount, etc.)
        for i, line in enumerate(lines):
            line = line.strip()
            if not line or len(line) < 10:
                continue
                
            # Skip header lines
            if any(header in line.lower() for header in ['s/n', 'name', 'sales order', 'amount', 'showroom']):
                continue
            
            # Try to extract sales data using regex patterns
            # Pattern for: Number Name SalesOrder Amount [other fields] ReasonCode
            pattern = r'(\d+)\s+([A-Za-z/]+)\s+(1-\d+)\s+(\$?\d+(?:,\d+)*\.?\d*)'
            match = re.search(pattern, line)
            
            if match:
                s_n, name, sales_order, amount = match.groups()
                
                # Clean amount
                amount = re.sub(r'[^\d.]', '', amount)
                
                # Look for reason code in the line or nearby lines
                reason_code = self.extract_reason_code(line, lines, i)
                
                sales_data.append({
                    'name': name.upper(),
                    'sales_order_no': sales_order,
                    'amount': amount,
                    'reason_code': reason_code
                })
        
        return sales_data

In [28]:
def extract_reason_code(self, current_line, all_lines, line_index):
        """Extract reason code from current line or nearby lines"""
        # Common patterns for reason codes
        reason_patterns = [
            r'\b(F|FB|M|LIVE|R1|R2|W)\b',
            r'\b(facebook|marketing|livestream|repeat|referral|walk-?in)\b'
        ]
        
        # Check current line and next few lines
        lines_to_check = [current_line]
        if line_index + 1 < len(all_lines):
            lines_to_check.append(all_lines[line_index + 1])
        if line_index + 2 < len(all_lines):
            lines_to_check.append(all_lines[line_index + 2])
        
        for line in lines_to_check:
            for pattern in reason_patterns:
                matches = re.findall(pattern, line, re.IGNORECASE)
                if matches:
                    code = matches[0].lower().strip()
                    if code in self.reason_mapping:
                        return self.reason_mapping[code]
                    else:
                        # Add to unclear markings for manual review
                        self.unclear_markings.add(code)
                        return f"UNCLEAR: {code}"
        
        return "Not specified"

In [29]:
def process_image(self, image_path):
        """Process a single image and extract sales data"""
        print(f"Processing: {image_path}")
        
        # Extract text from image
        text = self.extract_text_from_image(image_path)
        
        if not text:
            print(f"No text extracted from {image_path}")
            return []
        
        # Parse date and day
        date, day_text, day_num = self.parse_date_and_day(text)
        
        # Parse sales data
        sales_data = self.parse_sales_data(text)
        
        # Add date and day info to each record
        for record in sales_data:
            record['date'] = date or "Unknown"
            record['day'] = day_text or "Unknown"
            record['day_number'] = day_num or 0
            record['source_file'] = os.path.basename(image_path)
        
        return sales_data

In [30]:
def process_image(self, image_path):
        """Process a single image and extract sales data"""
        print(f"Processing: {image_path}")
        
        try:
            # Extract text from image
            text = self.extract_text_from_image(image_path)
            
            if not text:
                print(f"No text extracted from {image_path}")
                return []
            
            # Parse date and day
            date, day_text, day_num = self.parse_date_and_day(text)
            
            # Parse sales data
            sales_data = self.parse_sales_data(text)
            
            # Add date and day info to each record
            for record in sales_data:
                record['date'] = date or "Unknown"
                record['day'] = day_text or "Unknown"
                record['day_number'] = day_num or 0
                record['source_file'] = os.path.basename(image_path)
            
            return sales_data
            
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            return []

In [31]:
def process_multiple_images(self, image_paths):
        """Process multiple images and combine results"""
        all_data = []
        
        for image_path in image_paths:
            data = self.process_image(image_path)
            all_data.extend(data)
        
        self.extracted_data = all_data
        return all_data

In [32]:
def export_to_csv(self, output_path):
        """Export extracted data to CSV"""
        if not self.extracted_data:
            print("No data to export")
            return False
        
        df = pd.DataFrame(self.extracted_data)
        
        # Reorder columns
        column_order = ['date', 'day', 'day_number', 'name', 'sales_order_no', 'amount', 'reason_code', 'source_file']
        df = df[column_order]
        
        # Export to CSV
        df.to_csv(output_path, index=False)
        print(f"Data exported to: {output_path}")
        
        # Print unclear markings summary
        if self.unclear_markings:
            print("\nUnclear markings found (please clarify):")
            for marking in sorted(self.unclear_markings):
                print(f"  - {marking}")
        
        return True

In [33]:
class SalesExtractorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Sales Report OCR Extractor")
        self.root.geometry("800x600")
        
        self.extractor = SalesReportExtractor()
        self.selected_files = []
        self.create_widgets()

    def create_widgets(self):

        # Main frame
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Title
        title_label = ttk.Label(main_frame, text="Sales Report OCR Data Extractor", 
                               font=("Arial", 16, "bold"))
        title_label.grid(row=0, column=0, columnspan=2, pady=(0, 20))
        
        # File selection
        file_frame = ttk.LabelFrame(main_frame, text="Select Images", padding="10")
        file_frame.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=(0, 10))
        
        ttk.Button(file_frame, text="Select Images", 
                  command=self.select_files).grid(row=0, column=0, padx=(0, 10))
        
        self.file_count_label = ttk.Label(file_frame, text="No files selected")
        self.file_count_label.grid(row=0, column=1)
        
        # File list
        self.file_listbox = tk.Listbox(file_frame, height=8)
        self.file_listbox.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=(10, 0))
        
        # Process button
        self.process_btn = ttk.Button(main_frame, text="Process Images", 
                                     command=self.process_images, state="disabled")
        self.process_btn.grid(row=2, column=0, columnspan=2, pady=(10, 0))
        
        # Progress bar
        self.progress = ttk.Progressbar(main_frame, mode='indeterminate')
        self.progress.grid(row=3, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=(10, 0))
        
        # Results frame
        results_frame = ttk.LabelFrame(main_frame, text="Results", padding="10")
        results_frame.grid(row=4, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=(10, 0))
        
        self.results_text = tk.Text(results_frame, height=10, width=80)
        scrollbar = ttk.Scrollbar(results_frame, orient="vertical", command=self.results_text.yview)
        self.results_text.configure(yscrollcommand=scrollbar.set)
        
        self.results_text.grid(row=0, column=0, sticky=(tk.W, tk.E))
        scrollbar.grid(row=0, column=1, sticky=(tk.N, tk.S))
        
        # Export button
        self.export_btn = ttk.Button(main_frame, text="Export to CSV", 
                                    command=self.export_data, state="disabled")
        self.export_btn.grid(row=5, column=0, columnspan=2, pady=(10, 0))
        
        # Configure grid weights
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(4, weight=1)
        file_frame.columnconfigure(1, weight=1)
        results_frame.columnconfigure(0, weight=1)

    def select_files(self):
        file_types = [
            ("Image files", "*.png *.jpg *.jpeg *.bmp *.tiff *.tif"),
            ("All files", "*.*")
        ]
        
        files = filedialog.askopenfilenames(
            title="Select sales report images",
            filetypes=file_types
        )
        
        if files:
            self.selected_files = list(files)
            self.file_count_label.config(text=f"{len(files)} files selected")
            
            # Update listbox
            self.file_listbox.delete(0, tk.END)
            for file in files:
                self.file_listbox.insert(tk.END, os.path.basename(file))
            
            self.process_btn.config(state="normal")

    def process_images(self):
        if not self.selected_files:
            messagebox.showwarning("Warning", "Please select images first")
            return
        
        self.progress.start()
        self.process_btn.config(state="disabled")
        self.results_text.delete(1.0, tk.END)

        try:
            data = self.extractor.process_multiple_images(self.selected_files)

            # Display results
            self.results_text.insert(tk.END, f"Processing completed!\n")
            self.results_text.insert(tk.END, f"Total records extracted: {len(data)}\n\n")

            if data:
                self.results_text.insert(tk.END, "Sample data:\n")
                for i, record in enumerate(data[:5]):  # Show first 5 records
                    self.results_text.insert(tk.END, f"{i+1}. {record}\n")
                
                if len(data) > 5:
                    self.results_text.insert(tk.END, f"... and {len(data)-5} more records\n")
                
                self.export_btn.config(state="normal")
            else:
                self.results_text.insert(tk.END, "No data could be extracted from the images.\n")
                self.results_text.insert(tk.END, "Please check if the images are clear and contain the expected format.\n")
            
            # Show unclear markings
            if self.extractor.unclear_markings:
                self.results_text.insert(tk.END, f"\nUnclear markings found (please clarify):\n")
                for marking in sorted(self.extractor.unclear_markings):
                    self.results_text.insert(tk.END, f"  - {marking}\n")
            
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred during processing: {str(e)}")
            self.results_text.insert(tk.END, f"Error: {str(e)}\n")
        
        finally:
            # Stop progress bar
            self.progress.stop()
            self.process_btn.config(state="normal")

    def export_data(self):
        if not self.extractor.extracted_data:
            messagebox.showwarning("Warning", "No data to export")
            return
        
        output_file = filedialog.asksaveasfilename(
            title="Save CSV file",
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if output_file:
            success = self.extractor.export_to_csv(output_file)
            if success:
                messagebox.showinfo("Success", f"Data exported successfully to:\n{output_file}")
                

In [34]:
def main():
    # Check if required packages are installed
    try:
        import pytesseract
        import cv2
        import pandas as pd
    except ImportError as e:
        print(f"Required package not found: {e}")
        print("Please install required packages:")
        print("pip install pytesseract opencv-python pandas pillow")
        return
    
    root = tk.Tk()
    app = SalesExtractorGUI(root)
    root.mainloop()

In [35]:
if __name__ == "__main__":
    main()

KeyboardInterrupt: 